# SACR Tool — Complete Sentiment Analysis Pipeline

A comprehensive notebook covering all 4 phases:
1. **Data Preprocessing & EDA** — Load, clean, explore, visualize
2. **Feature Engineering** — Text vectorization, label creation
3. **Model Selection** — Train multiple classifiers
4. **Model Evaluation & XAI** — Confusion matrix, ROC-AUC, False Positive/Negative analysis, feature importance

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

ModuleNotFoundError: No module named 'pandas'

---
## Phase 1: Data Preprocessing & EDA

In [ ]:
# --- Load Dataset ---
DATA_PATH = r'D:\research\dataset\IMDB-Dataset.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# --- Basic Info & Missing Values ---
print('=== Info ===')
df.info()
print(f'\n=== Missing Values ===\n{df.isnull().sum()}')

In [ ]:
# --- Detect text & sentiment columns automatically ---
text_col = None
for col in df.columns:
    if df[col].dtype == 'object':
        avg_len = df[col].astype(str).str.len().mean()
        if avg_len > 50:
            text_col = col
            break

potential_sentiment_cols = [col for col in df.columns if 'sentiment' in col.lower() or 'label' in col.lower()]
rating_cols = [col for col in df.columns if 'rating' in col.lower() or 'score' in col.lower()]

print(f'Text column detected: {text_col}')
print(f'Potential sentiment columns: {potential_sentiment_cols}')
print(f'Rating/score columns: {rating_cols}')

In [ ]:
# --- Drop rows with null text ---
before = len(df)
df = df.dropna(subset=[text_col])
print(f'Dropped {before - len(df)} rows with null text')

In [ ]:
# --- Text Length Distribution ---
df['text_length'] = df[text_col].astype(str).apply(len)
df['word_count'] = df[text_col].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['text_length'], bins=50, edgecolor='black')
axes[0].set_title('Text Length Distribution')
axes[0].set_xlabel('Length (chars)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=50, edgecolor='black', color='orange')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# --- Class Distribution (if rating column exists) ---
if rating_cols:
    target_col = rating_cols[0]
    print(f'\n=== {target_col} Distribution ===')
    print(df[target_col].value_counts().sort_index())
    
    fig, ax = plt.subplots(figsize=(6, 4))
    df[target_col].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title('Class Distribution')
    ax.set_xlabel(target_col)
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Word Cloud for positive vs negative classes ---
from wordcloud import WordCloud

if rating_cols:
    target_col = rating_cols[0]
    pos_text = ' '.join(df[df[target_col] == 1][text_col].astype(str).head(500))
    neg_text = ' '.join(df[df[target_col] == 0][text_col].astype(str).head(500))
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    wc_pos = WordCloud(width=500, height=400, background_color='white',
                       colormap='Greens', max_words=100).generate(pos_text)
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].axis('off')
    axes[0].set_title('Positive Reviews')
    
    wc_neg = WordCloud(width=500, height=400, background_color='white',
                       colormap='Reds', max_words=100).generate(neg_text)
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title('Negative Reviews')
    plt.tight_layout()
    plt.show()

---
## Phase 2: Feature Engineering

Includes text cleaning, stopword customization (keep 'not' for sentiment, remove modals), contraction expansion, lemmatization, label creation, and vectorization.

In [ ]:
# --- Customized Stopwords ---
# Keep 'not' (critical for sentiment), remove modal verbs that don't carry sentiment
stop_words = stopwords.words('english')
new_stopwords = ['would', 'shall', 'could', 'might']
stop_words.extend(new_stopwords)
stop_words.remove('not')
stop_words = set(stop_words)
print(f'Custom stopwords loaded ({len(stop_words)} words)')

In [ ]:
# --- Contraction Expansion ---
def contraction_expansion(content):
    content = re.sub(r"won\'t", "would not", content)
    content = re.sub(r"can\'t", "can not", content)
    content = re.sub(r"don\'t", "do not", content)
    content = re.sub(r"shouldn\'t", "should not", content)
    content = re.sub(r"needn\'t", "need not", content)
    content = re.sub(r"hasn\'t", "has not", content)
    content = re.sub(r"haven\'t", "have not", content)
    content = re.sub(r"weren\'t", "were not", content)
    content = re.sub(r"mightn\'t", "might not", content)
    content = re.sub(r"didn\'t", "did not", content)
    content = re.sub(r"n\'t", " not", content)
    return content

# --- Text Cleaning Pipeline ---
def remove_special_chars(content):
    return re.sub(r'\W+', ' ', content)

def remove_urls(content):
    return re.sub(r'http\S+', '', content)

def remove_stopwords_from_text(content):
    clean = []
    for i in content.split():
        if i.strip().lower() not in stop_words and i.strip().lower().isalpha():
            clean.append(i.strip().lower())
    return ' '.join(clean)

def data_cleaning(content):
    content = contraction_expansion(content)
    content = remove_special_chars(content)
    content = remove_urls(content)
    content = remove_stopwords_from_text(content)
    return content

In [ ]:
# --- Apply Cleaning ---
print('Cleaning text...')
df['processed_text'] = df[text_col].astype(str).apply(data_cleaning)
df[['processed_text']].head(10)

In [ ]:
# --- Binary Labeling ---
# Option A: Use rating column (map 7-10 -> positive, 1-4 -> negative, remove 5-6 neutral)
# Option B: Keyword-based labeling if no rating column

if rating_cols:
    target_col = rating_cols[0]
    # Check if rating values are in a range (e.g. 1-10) or already binary
    if df[target_col].nunique() == 2 and set(df[target_col].unique()) == {0, 1}:
        df['label'] = df[target_col].astype(int)
        print('Using existing 0/1 labels.')
    elif df[target_col].max() > 1:
        # Map 0-4 or 1-4 -> negative (0), 7-10 -> positive (1), drop 5-6 neutral
        df['Label_temp'] = df[target_col].apply(lambda x: 1 if x >= 7 else (0 if x <= 4 else 2))
        df = df[df['Label_temp'] < 2]
        df['label'] = df['Label_temp'].astype(int)
        df.drop(columns=['Label_temp'], inplace=True)
        print('Labeled: rating>=7 -> positive, rating<=4 -> negative, 5-6 dropped')
    else:
        med = df[target_col].median()
        df['label'] = np.where(df[target_col] > med, 1, 0)
        print(f'Binarized {target_col} at median={med}')
else:
    df['label'] = np.where(
        df['processed_text'].str.contains(r'\b(good|excellent|positive|amazing|great|wonderful)\b', case=False, na=False),
        1, 0
    )
    print('Used keyword-based labeling (no rating column found).')

print(f'\nLabel distribution:\n{df["label"].value_counts().to_dict()}')
df[['processed_text', 'label']].head(10)

In [ ]:
# --- Drop rows with NaN ---
clean_df = df.dropna(subset=['processed_text', 'label']).copy()
print(f'Clean shape: {clean_df.shape}')

In [ ]:
# --- LemmaTokenizer (lemmatizes during vectorization) ---
lemmatizer = WordNetLemmatizer()

class LemmaTokenizer(object):
    def __init__(self):
        self.wordnetlemma = WordNetLemmatizer()
    def __call__(self, reviews):
        return [self.wordnetlemma.lemmatize(word) for word in word_tokenize(reviews)]

# --- Vectorizer Settings ---
VECTORIZER_TYPE = 'TF-IDF'  # or 'CountVectorizer'
NGRAM_MIN, NGRAM_MAX = 1, 3
MIN_DF = 10
MAX_FEATURES = 10000

if VECTORIZER_TYPE == 'CountVectorizer':
    vect = CountVectorizer(analyzer='word', tokenizer=LemmaTokenizer(),
                           ngram_range=(NGRAM_MIN, NGRAM_MAX), min_df=MIN_DF, max_features=MAX_FEATURES)
else:
    vect = TfidfVectorizer(analyzer='word', tokenizer=LemmaTokenizer(),
                            ngram_range=(NGRAM_MIN, NGRAM_MAX), min_df=MIN_DF, max_features=MAX_FEATURES)

X = vect.fit_transform(clean_df['processed_text'])
y = clean_df['label'].values
print(f'Vectorized shape: {X.shape}')
print(f'Labels: {np.bincount(y)}')

In [ ]:
# --- Train/Test Split ---
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'x_train: {x_train.shape}, x_test: {x_test.shape}')

In [ ]:
# --- Top Features by Score ---
feature_names = vect.get_feature_names_out()
if VECTORIZER_TYPE == 'CountVectorizer':
    feature_scores = np.asarray(X.sum(axis=0)).flatten()
    score_type = 'Frequency'
else:
    feature_scores = np.asarray(X.mean(axis=0)).flatten()
    score_type = 'Mean TF-IDF'

score_df = pd.DataFrame({'Feature': feature_names, 'Score': feature_scores})\
    .sort_values('Score', ascending=False).head(20)
print(f'Top 20 Features by {score_type}')
score_df

In [ ]:
# --- Chi-Squared Feature Selection ---
from sklearn.feature_selection import chi2
chi_scores, _ = chi2(x_train, y_train)
chi_df = pd.DataFrame({'Feature': feature_names, 'Chi2 Score': chi_scores})\
    .sort_values('Chi2 Score', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Chi2 Score', y='Feature', data=chi_df, ax=ax)
ax.set_title('Top 20 Features by Chi-Squared Score')
plt.tight_layout()
plt.show()

---
## Phase 3: Model Selection

Train multiple classifiers and collect metrics.

In [ ]:
# --- Model Configuration ---
SEED = 42
classifiers = {
    'Logistic Regression': LogisticRegression(C=10, max_iter=200, solver='lbfgs', random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=2, criterion='gini', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=2, random_state=SEED),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=SEED),
    'Naive Bayes': MultinomialNB(alpha=1.0)
}

results = []
trained_models = {}

In [ ]:
for name, clf in classifiers.items():
    print(f'\n{"="*50}\nTraining {name}...')
    start = time.time()
    
    clf.fit(x_train, y_train)
    training_time = time.time() - start
    
    y_pred = clf.predict(x_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'Training_Time': training_time
    })
    trained_models[name] = clf
    
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1:        {f1:.4f}')
    print(f'  Time:      {training_time:.2f}s')
    print()
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# --- Results Summary ---
results_df = pd.DataFrame(results).sort_values('F1_Score', ascending=False)
results_df

In [ ]:
# --- Performance Comparison Charts ---
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1_Score']
for i, metric in enumerate(metrics):
    sns.barplot(x='Model', y=metric, data=results_df, ax=axes[i])
    axes[i].set_title(f'{metric} Comparison')
    axes[i].tick_params(axis='x', rotation=45)
    for j, v in enumerate(results_df[metric]):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Training Time Comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='Model', y='Training_Time', data=results_df, ax=ax, palette='viridis')
ax.set_title('Training Time Comparison')
ax.set_ylabel('Time (seconds)')
for i, v in enumerate(results_df['Training_Time']):
    ax.text(i, v + 0.01, f'{v:.2f}s', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## Phase 4: Model Evaluation & XAI

Detailed evaluation: Confusion Matrix, ROC-AUC, False Positive / False Negative analysis, and Feature Importance.

In [ ]:
# --- Pick best model for detailed evaluation ---
best_model_name = results_df.iloc[0]['Model']
best_clf = trained_models[best_model_name]
y_pred_best = best_clf.predict(x_test)
print(f'Best model: {best_model_name}')
print(f'\n=== Detailed Classification Report ===')
report = classification_report(y_test, y_pred_best, output_dict=True)
pd.DataFrame(report).transpose()

In [ ]:
# --- Custom Confusion Matrix with annotations ---
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap='Blues')
    ax.grid(False)
    ax.xaxis.set(ticks=(0, 1), ticklabels=('Predicted 0 (Neg)', 'Predicted 1 (Pos)'))
    ax.yaxis.set(ticks=(0, 1), ticklabels=('Actual 0 (Neg)', 'Actual 1 (Pos)'))
    ax.set_ylim(1.5, -0.5)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='darkred', fontsize=16, fontweight='bold')
    plt.title(title)
    plt.show()

plot_confusion_matrix(y_test, y_pred_best, f'Confusion Matrix — {best_model_name}')

In [ ]:
# --- ROC-AUC Curve ---
if hasattr(best_clf, 'predict_proba'):
    y_prob = best_clf.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'ROC curve (AUC = {auc_score:.4f})', lw=2)
    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {best_model_name}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'AUC Score: {auc_score:.4f}')
else:
    print(f'{best_model_name} does not support probability predictions.')

In [ ]:
# --- Confusion Matrices for ALL models ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, (name, clf) in enumerate(trained_models.items()):
    y_pred = clf.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Neg', 'Pos']).plot(ax=axes[i])
    axes[i].set_title(name)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- False Positive & False Negative Analysis ---
y_pred_all = best_clf.predict(x_test)

# Get original text (non-vectorized) for inspection
# We need the test indices from the split
train_idx, test_idx = train_test_split(np.arange(len(clean_df)), test_size=0.3, random_state=42)
test_texts = clean_df.iloc[test_idx]['processed_text'].values
test_labels = clean_df.iloc[test_idx]['label'].values

fp_indices = np.where((y_pred_all == 1) & (y_test == 0))[0]
fn_indices = np.where((y_pred_all == 0) & (y_test == 1))[0]

print(f'\n=== False Positives (predicted positive, actual negative): {len(fp_indices)} ===')
for idx in fp_indices[:10]:
    text_preview = test_texts[idx][:200]
    print(f'  [{idx}] {text_preview}...')

print(f'\n=== False Negatives (predicted negative, actual positive): {len(fn_indices)} ===')
for idx in fn_indices[:10]:
    text_preview = test_texts[idx][:200]
    print(f'  [{idx}] {text_preview}...')

In [ ]:
# --- Feature Importance (Tree-based models) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

tree_models = [('Random Forest', trained_models.get('Random Forest')),
               ('Decision Tree', trained_models.get('Decision Tree')),
               ('AdaBoost', trained_models.get('AdaBoost')),
               ('Logistic Regression', trained_models.get('Logistic Regression'))]

ax_idx = 0
for name, clf in tree_models:
    if clf is None:
        continue
    if hasattr(clf, 'coef_'):
        importances = np.abs(clf.coef_[0])
    elif hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    else:
        continue

    top_n = 15
    top_idx = np.argsort(importances)[-top_n:]

    axes[ax_idx].barh(range(top_n), importances[top_idx], color='steelblue')
    axes[ax_idx].set_yticks(range(top_n))
    axes[ax_idx].set_yticklabels([feature_names[i] for i in top_idx])
    axes[ax_idx].set_title(f'Top {top_n} Features — {name}')
    axes[ax_idx].invert_yaxis()
    ax_idx += 1

plt.tight_layout()
plt.show()

In [ ]:
# --- Custom Input Prediction ---
def clean_single_text(text):
    if not isinstance(text, str):
        return ''
    text = contraction_expansion(text)
    text = remove_special_chars(text)
    text = remove_urls(text)
    text = remove_stopwords_from_text(text)
    return text

user_input = input('Enter a review to analyze: ')
if user_input.strip():
    cleaned = clean_single_text(user_input)
    vec = vect.transform([cleaned])
    pred = best_clf.predict(vec)[0]
    label = 'Positive' if pred == 1 else 'Negative'

    if hasattr(best_clf, 'predict_proba'):
        proba = best_clf.predict_proba(vec)[0]
        confidence = proba[int(pred)]
    else:
        confidence = None

    print(f'\n{"="*50}')
    print(f'Review: {user_input}')
    print(f'Cleaned: {cleaned}')
    print(f'Prediction: {label}')
    if confidence:
        print(f'Confidence: {confidence:.2%}')
        print(f'{"="*50}')
else:
    print('No input entered.')


In [ ]:
# --- Compare predictions across all models ---
user_input = input('\nEnter another review (or press Enter to skip): ') if 'user_input' not in dir() else user_input
if user_input and user_input.strip():
    cleaned = clean_single_text(user_input)
    vec = vect.transform([cleaned])
    print(f'{"Model":<25} {"Prediction":<15} {"Confidence":<10}')
    print('-' * 50)
    for name, clf in trained_models.items():
        p = clf.predict(vec)[0]
        lbl = 'Positive' if p == 1 else 'Negative'
        if hasattr(clf, 'predict_proba'):
            conf = clf.predict_proba(vec)[0][int(p)]
            print(f'{name:<25} {lbl:<15} {conf:.2%}')
        else:
            print(f'{name:<25} {lbl:<15} N/A')
else:
    print('Skipped.')


---
*Notebook generated from SACR Tool pipeline — all 4 phases in one end-to-end flow.*